# Acquire the first TEDx candidate

Candidate: [Urdu-Mera Payaam-e-Muhabbat | Dr. Arfa Zehra | TEDxUCP](https://www.youtube.com/watch?v=VD-RK-nY66k), official TEDx Talks channel. Duration reported by YouTube: 17:21.

This notebook downloads the full source for inspection and records provenance. It does not transcribe, crop, normalize, or approve the video as dataset material. No GPU is needed for acquisition. Acquisition executed successfully in Colab on 2026-09-26. [Saved run](https://colab.research.google.com/drive/1brxsXxNMf_xJ-cyAjllmu-1CJvswAbTz). This verifies downloading and a short decode check, not transcript or synchronization accuracy.

TED's [usage policy](https://www.ted.com/about/our-organization/our-policies-terms/ted-talks-usage-policy) restricts derivatives and unauthorized dubbing. Download availability is not dataset redistribution permission. Rights for the planned corpus remain unresolved. YouTube reports the audio language as `hi`; confirm the actual speech and record Urdu/code-switching after listening.

Run cells in order. Colab runtime storage is temporary. If YouTube refuses the request, use an authorized source file; do not paste account cookies or credentials into the notebook.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yt-dlp[default]", "imageio-ffmpeg"], check=True)


In [ ]:
from pathlib import Path
import json, shutil, hashlib
from datetime import datetime, timezone
from importlib.metadata import version
import imageio_ffmpeg
from yt_dlp import YoutubeDL

URL = "https://www.youtube.com/watch?v=VD-RK-nY66k"
ROOT = Path("/content/urdu-av-pilot")
RAW = ROOT / "raw"
RAW.mkdir(parents=True, exist_ok=True)
BIN = ROOT / "tools"
BIN.mkdir(exist_ok=True)
FFMPEG = BIN / "ffmpeg"
if not FFMPEG.exists():
    FFMPEG.symlink_to(imageio_ffmpeg.get_ffmpeg_exe())
COMMON = {"noplaylist": True, "socket_timeout": 20, "retries": 1, "extractor_retries": 1}
if shutil.which("node"):
    COMMON["js_runtimes"] = {"node": {}}
print({"yt_dlp": version("yt-dlp"), "imageio_ffmpeg": version("imageio-ffmpeg"), "folder": str(ROOT)})


In [ ]:
with YoutubeDL({**COMMON, "quiet": True}) as ydl:
    info = ydl.extract_info(URL, download=False)
manifest = {
    "source_id": info["id"], "source_url": URL, "title": info.get("title"),
    "channel": info.get("channel"), "duration_seconds": info.get("duration"),
    "platform_language": info.get("language"), "verified_spoken_language": None,
    "dataset_reuse_permission": "not established",
    "review_status": "pending audio, visual, and sync review",
    "pilot_interval": None, "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "tool_versions": {"yt-dlp": version("yt-dlp"), "imageio-ffmpeg": version("imageio-ffmpeg")},
}
print(json.dumps(manifest, ensure_ascii=False, indent=2))
# Save selected metadata only, not expiring stream URLs or account information.
(RAW / "source_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


## Download the observed 1080p video and audio streams

Formats `137` (1080p H.264 video) and `140` (AAC audio without the DRC suffix) were listed for this source during inspection. They may change. If unavailable, inspect the current format list before selecting a replacement; do not silently use an audio-only or low-resolution file. This cell remuxes the streams into one MP4 without changing their playback speed.


In [ ]:
opts = {
    **COMMON, "format": "137+140", "merge_output_format": "mp4",
    "ffmpeg_location": str(FFMPEG),
    "outtmpl": str(RAW / "%(id)s.%(ext)s"),
}
with YoutubeDL(opts) as ydl:
    ydl.download([URL])
VIDEO = RAW / f"{manifest['source_id']}.mp4"
if not VIDEO.is_file():
    raise RuntimeError("Merged MP4 was not created. Inspect the download/merge error above.")
# Decode a short interval to check that the file has readable audio and video.
subprocess.run([str(FFMPEG), "-v", "error", "-i", str(VIDEO), "-t", "2",
                "-map", "0:v:0", "-map", "0:a:0", "-f", "null", "-"], check=True)
hash_value = hashlib.sha256()
with VIDEO.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
        hash_value.update(chunk)
manifest.update({"local_filename": VIDEO.name, "size_bytes": VIDEO.stat().st_size,
                 "sha256": hash_value.hexdigest(), "download_status": "complete"})
(RAW / "source_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved {VIDEO} ({VIDEO.stat().st_size / 1024**2:.1f} MiB)")


## Review before cutting

Open the original YouTube link and inspect the downloaded source as needed. Choose about five minutes of speech after checking language, visible mouth detail, shot changes, and synchronization. Exclude opening graphics, audience shots, applause, and unusable views. A single stage speaker does not guarantee continuous usable mouth footage. No exact five-minute interval has been approved yet.

Keep the source unchanged and save the manifest before the Colab runtime ends. To download both files to your computer, run the optional cell below; this may take time.


In [ ]:
# Optional: uncomment to save the source and its manifest to your computer.
# from google.colab import files
# files.download(str(VIDEO))
# files.download(str(RAW / "source_manifest.json"))


# Read back the saved acquisition record.
saved = json.loads((RAW / "source_manifest.json").read_text())
assert saved["download_status"] == "complete"
assert (RAW / saved["local_filename"]).stat().st_size == saved["size_bytes"]
print(json.dumps(saved, ensure_ascii=False, indent=2))
